In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read.parquet("data/external_table/part-00000-6109de51-a0e8-41af-839d-58e5ef769898-c000.snappy.parquet", inferSchema=True, header=True)

In [3]:
orders_df.createOrReplaceTempView('orders')

### 5. customers who placed atleast one order

In [4]:
# First Let's see how many customers placed no order at all
from pyspark.sql.functions import countDistinct
orders_agg = orders_df.groupBy('customer_id').agg(countDistinct('order_id').alias('order_count'))
orders_agg.show(10)

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|       8638|          8|
|      11317|          7|
|      11458|          7|
|       2142|          4|
|      11858|          7|
|      10362|          6|
|       1580|          5|
|       4935|          7|
|       3794|          5|
|       6397|          8|
+-----------+-----------+
only showing top 10 rows



In [5]:
orders_agg.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- order_count: long (nullable = false)



In [6]:
orders_agg.filter("order_count < 1").show(10)

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
+-----------+-----------+



#### Using 'col'

```
orders_df.groupBy('customer_id') \
         .agg(count('order_id').alias('order_count')) \
         .filter(col('order_count') > 1) \
         .show(10)
```

In [7]:
spark.sql("""
    SELECT customer_id, COUNT(order_id) as order_count
    FROM orders
    GROUP BY customer_id
    HAVING order_count < 1
    ORDER BY order_count DESC
    LIMIT 10
""")

customer_id,order_count


### filtering after an aggregation in SQL requires the HAVING keyword, not WHERE

In [8]:
spark.stop()